# Lab 08-02: Score Responses With Custom Autoraters

**Book chapter:** Chapter 08: Evaluating Agents  
**Estimated time:** 20 minutes

## Overview

Cymbal Retail's word-overlap scores did not match what reviewers cared about: whether
an answer applies the returns policy correctly and tells the shopper what to do next.
In this lab, you score four responses with ROUGE-1, a prebuilt rubric metric, and a
custom autorater for the returns policy, and you compare the rankings.

## Learning Objectives

By the end of this lab, you will be able to:

- Run the Gen AI evaluation service from Python
- Compare computation-based and LLM-judged metrics
- Define a custom autorater with criteria and a rating scale
- Use judge explanations to diagnose failures

## Before You Begin

- Complete **Lab 00: Set Up the Lab Environment** in the `Appendix-C-Setup` folder

**Cost:** A few cents for judge model requests. No resources are created.

## How to Use This Notebook

Run the cells in order, from top to bottom, by selecting each code cell and pressing **Shift+Enter**. Read the explanation above each cell before you run it, and compare your output with the expected result that the text describes.

## Key Concepts

| Metric type | Measures | Weakness |
|---|---|---|
| **Computation-based** (ROUGE) | Word overlap with a reference | Misses meaning |
| **Prebuilt LLM judge** | General quality | Not specific to your business |
| **Custom autorater** | Your own criteria, such as policy compliance | Must be calibrated against human ratings |

## Setup

In [ ]:
%pip install --quiet --upgrade "google-cloud-aiplatform[evaluation]" pandas

> **Note:** If you are using Google Colab and a message asks you to restart the
> session after the installation, click **Restart session**, and then continue
> from the next cell. You don't need to run the installation cell again.

In [ ]:
import os

# Replace with your lab project ID (see Appendix C).
PROJECT_ID = "your-project-id"  # @param {type:"string"}
# A current Gemini Flash model ID. Check the Google models page if this
# model has been retired.
MODEL = "gemini-3.8-flash"  # @param {type:"string"}

# Fall back to the environment variables set by labenv.sh (Appendix C).
if PROJECT_ID == "your-project-id":
    PROJECT_ID = os.environ.get("GOOGLE_CLOUD_PROJECT", PROJECT_ID)
MODEL = os.environ.get("GEMINI_MODEL", MODEL)
assert PROJECT_ID != "your-project-id", "Set PROJECT_ID to your project ID."
# The lab code reads these environment variables.
os.environ["GOOGLE_CLOUD_PROJECT"] = PROJECT_ID
os.environ["GEMINI_MODEL"] = MODEL
print("Project:", PROJECT_ID)
print("Model:  ", MODEL)

In [ ]:
import sys

if "google.colab" in sys.modules:
    # In Colab, sign in with the Google account that owns the lab project.
    from google.colab import auth
    auth.authenticate_user(project_id=PROJECT_ID)
    print("Signed in to Google Cloud from Colab.")
else:
    # In Jupyter, VS Code, or Workbench, the client libraries use Application
    # Default Credentials. If a later cell reports a credentials error, run
    # this command in a terminal and then rerun the cell:
    #   gcloud auth application-default login
    print("Using Application Default Credentials.")

## Step 1: Collect the responses to evaluate

The four responses to the same question are fixed so that everyone sees the same
comparison. In practice, they would come from different versions of your agent.

| Response | Decision | Wording | Next step |
|---|---|---|---|
| A | Correct | Like the reference | Yes |
| B | Correct | Different | Yes |
| C | **Wrong** | Like the reference | Yes |
| D | Correct | Different | **No** |

The notebook imports the Agent Platform SDK as `agentplatform`, which is part of
`google-cloud-aiplatform`. Older examples import the same client as `vertexai`.

In [ ]:
import os

import agentplatform  # The Agent Platform SDK in google-cloud-aiplatform
import pandas as pd
from agentplatform import types

PROJECT = os.environ["GOOGLE_CLOUD_PROJECT"]
client = agentplatform.Client(project=PROJECT, location="us-central1")

QUESTION = ("A Gold member wants to return unworn boots delivered 45 days "
            "ago. Can they?")
REFERENCE = ("Yes. Gold members can return unworn items within 60 days, "
             "so the boots can be returned. Start the return from the "
             "order page.")
dataset = pd.DataFrame({
    "prompt": [QUESTION] * 4,
    "reference": [REFERENCE] * 4,
    "response": [
        # A: Correct, and worded like the reference.
        "Yes. Gold members can return unworn items within 60 days, so "
        "these boots can be returned. Start the return from the order "
        "page.",
        # B: Correct, but worded differently.
        "Absolutely. As a Gold member you have two months for unworn "
        "items, so you're covered. Open your order and choose Return.",
        # C: Worded like the reference, but applies the wrong rule.
        "No. Unworn items can be returned within 30 days, so these boots "
        "cannot be returned. Contact support from the order page.",
        # D: Correct, but gives no next step.
        "Yes, that is allowed.",
    ],
})

## Step 2: Define a computation metric, a prebuilt rubric metric, and a

Three kinds of metric:

- `rouge_1` is **computation-based**: it counts overlapping words.
- `RubricMetric.GENERAL_QUALITY` is a **prebuilt LLM-judged** metric.
- `policy_compliance` is a **custom autorater**: `MetricPromptBuilder` turns your
  instruction, criteria, and rating scale into a prompt for a judge model.

In [ ]:
# custom autorater for Cymbal Retail's returns policy.
policy_compliance = types.LLMMetric(
    name="policy_compliance",
    prompt_template=types.MetricPromptBuilder(
        instruction="Evaluate a customer service response against Cymbal "
                    "Retail's returns policy: unworn items 30 days, Gold "
                    "members 60 days, defective items 12 months.",
        criteria={
            "Correct rule": "The response applies the right return window "
                            "and reaches the right decision.",
            "Next step": "The response tells the customer what to do "
                         "next.",
        },
        rating_scores={
            "2": "The decision is correct and a next step is given.",
            "1": "The decision is correct but no next step is given.",
            "0": "The decision is wrong.",
        },
    ),
)
METRICS = [types.Metric(name="rouge_1"),
           types.RubricMetric.GENERAL_QUALITY,
           policy_compliance]

## Step 3: Run the evaluation

Run the evaluation. LLM-judged metrics take longer than ROUGE because a judge model reads every response.

In [ ]:
result = client.evals.evaluate(dataset=dataset, metrics=METRICS)
print("Average score per metric:")
for summary in result.summary_metrics:
    print(f"  {summary.metric_name:18} {summary.mean_score}")

## Step 4: Compare the scores for each response

Compare the scores for each response.

In [ ]:
rows = []
for label, case in zip("ABCD", result.eval_case_results):
    scores = case.response_candidate_results[0].metric_results
    rows.append({"response": label,
                 **{name: score.score for name, score in scores.items()}})
print(pd.DataFrame(rows).to_string(index=False))

**What to look for:** B scores low on ROUGE but high on `policy_compliance`; C scores
higher on ROUGE but 0 on `policy_compliance`; D scores 1 because it gives no next step.

## Step 5: Read the judge's explanation for the wrong-rule response

Custom autoraters explain their scores, which makes failures actionable.

In [ ]:
judged = result.eval_case_results[2].response_candidate_results[0]
print(judged.metric_results["policy_compliance"].explanation)

**Expected result:** an explanation that response C applied the 30-day window instead of the 60-day Gold window.

## Step 6: Experiment

In Step 2, add a third criterion, such as a friendly tone, and rerun Steps 2 to 4. Before
relying on a custom autorater, compare its scores with human ratings on a sample.

## Check Your Understanding

**1. Why does response C score better on ROUGE than response B?**

<details><summary>Show answer</summary>

C reuses the reference's words even though its decision is wrong, while B is correct but worded differently. ROUGE measures words, not correctness.

</details>

**2. What makes a custom autorater trustworthy?**

<details><summary>Show answer</summary>

Specific, observable criteria, a clear rating scale, and calibration against human ratings on a labeled sample.

</details>

**3. When would you still use ROUGE?**

<details><summary>Show answer</summary>

When the exact wording matters, such as fixed legal text, or as a cheap check alongside meaning-based metrics.

</details>

## Summary

You compared a computation-based metric, a prebuilt rubric metric, and a custom autorater,
and saw that only the autorater captured whether the policy was applied correctly. This lab
created no resources.